In [ ]:
import os
from pyspark.sql import functions as F
from src.utils.config import *
from src.ingestion.aviation_edge import fetch_flight_history
from src.ingestion.loaders import load_json_to_spark_df
from src.utils.kafka import create_kafka_consumer, consume_kafka_messages
from src.utils.spark import get_spark

spark = get_spark()

bootstrap_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS')
ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

consumer_instance = create_kafka_consumer(
    client_id='weather_processor',
    bootstrap_servers=bootstrap_servers,
    topic_name='historical_weather_data',
)

In [ ]:
# Consume form Kafka topic and store in a list
all_flights_data = consume_kafka_messages(consumer_instance)

# 3. Parallelize the JSON string into a weather collection RDD
weather_collection = spark.sparkContext.parallelize(all_weather_data, 10)

# 4. Create an initial DataFrame to inspect schema
raw_weather_df = spark.read.json(weather_collection)
schema_fields = raw_weather_df.columns
raw_weather_df = raw_weather_df.withColumns(
  {"weather": F.array_join(F.col("weather.main"), ", "),
   "weather_desc": F.array_join(F.col("weather.description"), ", "),
   "dt": F.from_unixtime(F.col("dt")),
  }
)

# raw_weather_df.show(n=3, truncate=False)
# raw_weather_df.printSchema()

# 5. Define selection expressions with safety checks for 'rain' and 'snow'
select_exprs = [
    F.col("dt").alias("date_obs"),
    F.col("main.temp").alias("Current_Temp  (\N{DEGREE SIGN}C)"),
    F.col("main.feels_like").alias("Feels_Like  (\N{DEGREE SIGN}C)"),
    F.col("main.pressure").alias("Pressure (hPa)"),
    F.col("main.humidity").alias("Humidity (%)"),
    F.col("main.temp_min").alias("Min_Temp (\N{DEGREE SIGN}C)"),
    F.col("main.temp_max").alias("Max_Temp (\N{DEGREE SIGN}C)"),
    F.col("wind.speed").alias("Wind_Speed (m/s)"),
    F.col("wind.deg").alias("Wind_Deg"),
    F.col("clouds.all").alias("Cloudiness (%)"),
    # Check for rain
    F.col("rain.1h") if "rain" in schema_fields else F.lit(None).alias("Rainfall (mm)"),
    # Check for snow
    F.col("snow.1h") if "snow" in schema_fields else F.lit(None).alias("Snowfall (mm)"),
    F.col("weather").alias("Weather_Main"), # weather.main is things like rain, clouds, snow etc.
    F.col("Weather_Desc"), # details on weather like cloud formation etc.
]

raw_weather_df = raw_weather_df.select(*select_exprs).sort(F.asc("dt"))
raw_weather_df.show(n=1, truncate=False)